In [55]:
import pandas as pd
import numpy as np

In [56]:
def extract_pos_coordinates(filepath, columns_to_extract=None, output_csv=None):
    """
    Extract specified columns from a position file.
    
    Parameters:
    - filepath: Path to the input file
    - columns_to_extract: List of column indices to extract (0-based). 
                         If None, extracts first 4 columns by default.
    - output_csv: Optional path to save results as CSV
    """
    if columns_to_extract is None:
        columns_to_extract = [0, 1, 2, 3]  # Default: first 4 columns
    
    extracted_rows = []
    column_names = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip metadata and comments
            if not line or line.startswith('%'):
                continue
            
            # Extract and clean the comma-separated values
            parts = [p.strip() for p in line.split(',')]
            
            # Extract only the specified columns
            try:
                extracted_values = []
                for col_idx in columns_to_extract:
                    if col_idx < len(parts):
                        # Try to convert to float, otherwise keep as string
                        try:
                            extracted_values.append(float(parts[col_idx]))
                        except ValueError:
                            extracted_values.append(parts[col_idx])
                    else:
                        extracted_values.append(None)  # or '' for empty string
                
                extracted_rows.append(extracted_values)
            except (IndexError, ValueError) as e:
                print(f"Warning: Skipping line due to error: {e}")
                print(f"Line content: {line}")
                continue

    # Optional: write to CSV
    if output_csv:
        import csv
        # Generate column headers based on indices
        column_names = [f'Column_{idx}' for idx in columns_to_extract]
        
        with open(output_csv, 'w', newline='') as f_out:
            writer = csv.writer(f_out)
            writer.writerow(column_names)
            writer.writerows(extracted_rows)

    return extracted_rows

# Example usage
pos_file_path = r'F:\zizo\RTKCorrection\src\research\research_data\testKinematic.pos'



# Example 3: Extract with CSV output
output3 = extract_pos_coordinates(
    pos_file_path, 
    columns_to_extract=[0, 1, 2, 3],  # GPST, Latitude, Longitude
    output_csv='extracted_coordinates.csv'
)

pd.set_option('display.precision', 10)
df_y = pd.DataFrame(output3, columns=['GPST', 'x_y', 'y_y', 'z_y'])

In [57]:
df_y.tail()

,GPST,x_y,y_y,z_y
7905,2024/02/14 06:32:16.01,464115.5768,5.6326302641e+06,2.9489959568e+06
7906,2024/02/14 06:32:17.01,464120.9077,5.6326237236e+06,2.9489969789e+06
7907,2024/02/14 06:32:18.01,464112.4989,5.6326237124e+06,2.9490027718e+06
7908,2024/02/14 06:32:19.01,464113.8278,5.6326229910e+06,2.9490044138e+06
7909,2024/02/14 06:32:20.01,464128.4003,5.6326161349e+06,2.9489968973e+06


In [58]:
pos_file_path2 = r'F:\zizo\RTKCorrection\src\research\research_data\testsSPP.pos'
output3 = extract_pos_coordinates(
    pos_file_path2, 
    columns_to_extract=[0, 1, 2, 3, 5, 6, 7, 8],  # GPST, Latitude, Longitude
    output_csv='extracted_coordinates.csv'
)
df_x = pd.DataFrame(output3, columns=['GPST', 'x_x', 'y_x', 'z_x', 'ns','sdx(m)','sdy(m)','sdz(m)'])

In [59]:
df_x.tail()

,GPST,x_x,y_x,z_x,ns,sdx(m),sdy(m),sdz(m)
7968,2024/02/14 06:32:20.01,464124.0612,5.6326319059e+06,2.9490136174e+06,12.0,6.0030,21.8619,8.2872
7969,2024/02/14 06:32:21.01,464124.1856,5.6326333853e+06,2.9490157864e+06,12.0,6.0032,21.8594,8.2869
7970,2024/02/14 06:32:22.01,464124.7210,5.6326347517e+06,2.9490175730e+06,11.0,10.7966,30.3993,11.5897
7971,2024/02/14 06:32:23.01,464132.2236,5.6326295838e+06,2.9490205933e+06,11.0,10.7976,30.3961,11.5892
7972,2024/02/14 06:32:24.01,464116.7796,5.6326436905e+06,2.9490250465e+06,7.0,19.6735,34.7704,15.2285


# EDA

In [60]:
df = pd.merge(df_x, df_y, on='GPST', how='inner', suffixes=('_x', '_y'))

In [61]:
df.head()

,GPST,x_x,y_x,z_x,ns,sdx(m),sdy(m),sdz(m),x_y,y_y,z_y
0,2024/02/14 04:19:28.00,461332.1582,5.6344514771e+06,2.9459406828e+06,13.0,3.5287,11.2818,6.9751,461329.8229,5.6344394938e+06,2.9459295375e+06
1,2024/02/14 04:19:29.00,461332.0927,5.6344515642e+06,2.9459409229e+06,13.0,3.5287,11.2825,6.9752,461329.8225,5.6344394940e+06,2.9459295390e+06
2,2024/02/14 04:19:30.00,461332.1570,5.6344516425e+06,2.9459407543e+06,13.0,3.5287,11.2832,6.9753,461329.8234,5.6344394981e+06,2.9459295394e+06
3,2024/02/14 04:19:31.00,461332.1452,5.6344513007e+06,2.9459406677e+06,13.0,3.5287,11.2839,6.9755,461329.8237,5.6344394940e+06,2.9459295371e+06
4,2024/02/14 04:19:32.00,461332.2278,5.6344510727e+06,2.9459404919e+06,13.0,3.5287,11.2846,6.9756,461329.8229,5.6344394981e+06,2.9459295392e+06


In [62]:
df.isna().sum()

GPST      0
x_x       0
y_x       0
z_x       0
ns        0
sdx(m)    0
sdy(m)    0
sdz(m)    0
x_y       0
y_y       0
z_y       0
dtype: int64

In [63]:
print(df['x_x'].max() - df['x_x'].min())
print(df['y_x'].max() - df['y_x'].min())
print(df['z_x'].max() - df['z_x'].min())

6512.785900000017
4043.4101999998093
7958.7933999998495


In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7910 entries, 0 to 7909
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GPST    7910 non-null   object 
 1   x_x     7910 non-null   float64
 2   y_x     7910 non-null   float64
 3   z_x     7910 non-null   float64
 4   ns      7910 non-null   float64
 5   sdx(m)  7910 non-null   float64
 6   sdy(m)  7910 non-null   float64
 7   sdz(m)  7910 non-null   float64
 8   x_y     7910 non-null   float64
 9   y_y     7910 non-null   float64
 10  z_y     7910 non-null   float64
dtypes: float64(10), object(1)
memory usage: 679.9+ KB


# Feature Engineering

In [65]:

def feature_engineers(df):
    # calculate correction position
    df['x_y'] = (df['x_y'] - df['x_x']) 
    df['y_y']  = (df['y_y'] - df['y_x']) 
    df['z_y']    = df['z_y'] - df['z_x']

    # Convert GPST to datetime and calculate elapsed time in seconds
    df['GPST'] = pd.to_datetime(df['GPST'], format='%Y/%m/%d %H:%M:%S.%f')
    df['time_elapsed'] = (df['GPST'] - df['GPST'].iloc[0]).dt.total_seconds()

    # Compute speed = distance / time
    df['dx'] = df['x_x'].diff()
    df['dy'] = df['y_y'].diff()
    df['dz'] = df['z_y'].diff()
    df['dt'] = df['time_elapsed'].diff()
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2 + df['dz']**2)
    df['speed'] = df['distance'] / df['dt']
    df=df.drop(columns=['dx','dy','dz','distance','dt'],axis = 1)

    df['x_lag1'] = df['x_x'].shift(1)
    df['y_lag1'] = df['y_x'].shift(1)
    df['z_lag1'] = df['z_x'].shift(1)
    
    return df
df = feature_engineers(df)
df.head()
    

,GPST,x_x,y_x,z_x,ns,sdx(m),sdy(m),sdz(m),x_y,y_y,z_y,time_elapsed,speed,x_lag1,y_lag1,z_lag1
0,2024-02-14 04:19:28,461332.1582,5.6344514771e+06,2.9459406828e+06,13.0,3.5287,11.2818,6.9751,-2.3353,-11.9832999995,-11.1453000000,0.0,NaN,NaN,NaN,NaN
1,2024-02-14 04:19:29,461332.0927,5.6344515642e+06,2.9459409229e+06,13.0,3.5287,11.2825,6.9752,-2.2702,-12.0702000000,-11.3839000002,1.0,0.2622438183,461332.1582,5.6344514771e+06,2.9459406828e+06
2,2024-02-14 04:19:30,461332.1570,5.6344516425e+06,2.9459407543e+06,13.0,3.5287,11.2832,6.9753,-2.3336,-12.1443999996,-11.2149000000,2.0,0.1954510937,461332.0927,5.6344515642e+06,2.9459409229e+06
3,2024-02-14 04:19:31,461332.1452,5.6344513007e+06,2.9459406677e+06,13.0,3.5287,11.2839,6.9755,-2.3215,-11.8066999996,-11.1306000003,3.0,0.3482628605,461332.1570,5.6344516425e+06,2.9459407543e+06
4,2024-02-14 04:19:32,461332.2278,5.6344510727e+06,2.9459404919e+06,13.0,3.5287,11.2846,6.9756,-2.4049,-11.5745999999,-10.9526999998,4.0,0.3038775741,461332.1452,5.6344513007e+06,2.9459406677e+06
